# Experiment 3 — Loan Amount Prediction using Linear and Regularized Regression
ICS1512 Machine Learning Algorithms Laboratory

This notebook loads the real Kaggle *Loan Amount* dataset (`archive/train.csv`),
cleans it, explores it, and then trains and compares four regression models —
plain Linear Regression, Ridge, Lasso and Elastic Net — for predicting the
loan amount that gets sanctioned to an applicant.

In [1]:
import pandas as pd
import numpy as np
import time, pickle
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 130
RNG = 42
np.random.seed(RNG)

## 1. Loading the data
The raw file has 30,000 applicant records and 24 columns. `Customer ID`,
`Name` and `Property ID` are pure identifiers, so they are dropped straight
away — they carry no information a model could learn from.

In [2]:
df = pd.read_csv("archive/train.csv")
print("Raw shape:", df.shape)
df = df.drop(columns=["Customer ID", "Name", "Property ID"])
df.head()

Raw shape: (30000, 24)


,Gender,Age,Income (USD),Income Stability,Profession,Type of Employment,Location,Loan Amount Request (USD),Current Loan Expenses (USD),Expense Type 1,...,Dependents,Credit Score,No. of Defaults,Has Active Credit Card,Property Age,Property Type,Property Location,Co-Applicant,Property Price,Loan Sanction Amount (USD)
0,F,56,1933.05,Low,Working,Sales staff,Semi-Urban,72809.58,241.08,N,...,3.0,809.44,0,NaN,1933.05,4,Rural,1,119933.46,54607.18
1,M,32,4952.91,Low,Working,NaN,Semi-Urban,46837.47,495.81,N,...,1.0,780.40,0,Unpossessed,4952.91,2,Rural,1,54791.00,37469.98
2,F,65,988.19,High,Pensioner,NaN,Semi-Urban,45593.04,171.95,N,...,1.0,833.15,0,Unpossessed,988.19,2,Urban,0,72440.58,36474.43
3,F,65,NaN,High,Pensioner,NaN,Rural,80057.92,298.54,N,...,2.0,832.70,1,Unpossessed,NaN,2,Semi-Urban,1,121441.51,56040.54
4,F,31,2614.77,Low,Working,High skill tech staff,Semi-Urban,113858.89,491.41,N,...,NaN,745.55,1,Active,2614.77,4,Semi-Urban,1,208567.91,74008.28


## 2. Cleaning the target column
A handful of rows have a missing loan-sanction amount, and 338 rows carry a
`-999` sentinel value that clearly isn't a real loan amount. Both kinds of
rows are removed before anything else happens.

In [3]:
target = "Loan Sanction Amount (USD)"
df = df[df[target].notna()]
df = df[df[target] >= 0]
print("Shape after removing invalid target rows:", df.shape)
print("Fraction of applicants with a sanctioned amount of exactly 0:",
      round((df[target] == 0).mean(), 3))

Shape after removing invalid target rows: (29322, 21)
Fraction of applicants with a sanctioned amount of exactly 0: 0.268


In [4]:
numeric_cols = [
    "Age", "Income (USD)", "Loan Amount Request (USD)",
    "Current Loan Expenses (USD)", "Dependents", "Credit Score",
    "No. of Defaults", "Property Age", "Co-Applicant", "Property Price",
]
categorical_cols = [
    "Gender", "Income Stability", "Profession", "Type of Employment",
    "Location", "Expense Type 1", "Expense Type 2",
    "Has Active Credit Card", "Property Type", "Property Location",
]
df[numeric_cols + categorical_cols].isnull().sum().sort_values(ascending=False)

Type of Employment             7125
Property Age                   4673
Income (USD)                   4411
Dependents                     2402
Credit Score                   1641
Income Stability               1628
Has Active Credit Card         1521
Property Location               341
Current Loan Expenses (USD)     163
Gender                           51
Property Price                    0
Co-Applicant                      0
Profession                        0
No. of Defaults                   0
Location                          0
Expense Type 1                    0
Expense Type 2                    0
Loan Amount Request (USD)         0
Property Type                     0
Age                               0
dtype: int64

## 3. Exploratory Data Analysis
Before touching any model, it helps to look at how the target is shaped, how
much data is missing, how each feature relates to the target, and how the
numeric features relate to each other.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
sns.histplot(df[target], bins=50, kde=True, color="#4C9A78", ax=axes[0])
axes[0].set_title("Loan Sanction Amount - Distribution")
axes[0].set_xlabel("Loan Sanction Amount (USD)")
sns.boxplot(x=df[target], color="#4C9A78", ax=axes[1])
axes[1].set_title("Loan Sanction Amount - Spread")
axes[1].set_xlabel("Loan Sanction Amount (USD)")
plt.tight_layout()
plt.savefig("images/target_distribution.png", bbox_inches="tight")
plt.show()
print("Target skewness:", round(df[target].skew(), 3))

Target skewness: 1.222


/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/2592592203.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
miss = df[numeric_cols + categorical_cols].isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.6))
sns.barplot(x=miss.values, y=miss.index, color="#C0554A", ax=ax)
ax.set_xlabel("Number of missing entries")
ax.set_title("Missing Values by Column")
plt.tight_layout()
plt.savefig("images/missing_values.png", bbox_inches="tight")
plt.show()

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/1378894816.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
sample = df.sample(n=min(1500, len(df)), random_state=RNG)
fig, axes = plt.subplots(2, 5, figsize=(16, 6.4))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].scatter(sample[col], sample[target], s=6, alpha=0.35, color="#3E7CB1")
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel("Loan Amount" if i % 5 == 0 else "", fontsize=8)
    axes[i].tick_params(labelsize=7)
plt.suptitle("Predictor Features vs Target Loan Sanction Amount", y=1.02)
plt.tight_layout()
plt.savefig("images/feature_vs_target.png", bbox_inches="tight")
plt.show()

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/307050893.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
corr = df[numeric_cols + [target]].corr()
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"size": 7}, ax=ax)
ax.set_title("Correlation Heatmap - Numeric Features")
plt.tight_layout()
plt.savefig("images/correlation_heatmap.png", bbox_inches="tight")
plt.show()
corr[target].sort_values(ascending=False)

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/2899423685.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Loan Sanction Amount (USD)     1.000000
Loan Amount Request (USD)      0.735255
Property Price                 0.695797
Current Loan Expenses (USD)    0.491367
Credit Score                   0.372873
Income (USD)                   0.038270
Property Age                   0.037909
Dependents                     0.008505
Age                            0.006890
No. of Defaults               -0.004169
Co-Applicant                  -0.008557
Name: Loan Sanction Amount (USD), dtype: float64

**A data quality find worth flagging:** while checking pairwise correlations
between the numeric features themselves, `Income (USD)` and `Property Age`
turn out to be *identical* for every single row (correlation of exactly
1.00). This is almost certainly a labelling artifact in the source file
rather than a genuine relationship, and keeping both columns would hand the
model a perfectly duplicated feature. `Property Age` is dropped before
modelling, and `Loan Amount Request (USD)` and `Property Price` (r = 0.95)
are also worth keeping an eye on since they carry a lot of shared
information.

In [9]:
feat_corr = df[numeric_cols].corr().abs().copy()
vals = feat_corr.values.copy()
np.fill_diagonal(vals, 0)
feat_corr = pd.DataFrame(vals, index=feat_corr.index, columns=feat_corr.columns)
print("Most correlated feature pair:", feat_corr.stack().idxmax(), "-> r =", round(feat_corr.stack().max(), 4))
numeric_cols.remove("Property Age")

Most correlated feature pair: ('Income (USD)', 'Property Age') -> r = 1.0


## 4. Preprocessing
* Missing numeric values are filled with the column **median**.
* Missing categorical values are filled with an explicit `"Unknown"` label
  (rather than the most frequent category) since several columns are missing
  20%+ of their entries, and silently assigning them to the mode would be
  misleading.
* Categorical columns are converted to integer codes.
* An 80/20 train-test split is taken, and the training portion is split
  again (75/25) to get a validation slice used later for the Ridge
  train-vs-validation curve.
* Every feature is standardised (zero mean, unit variance) so the
  regularization penalty treats all coefficients on the same footing.

In [10]:
X = df[numeric_cols + categorical_cols].copy()
y = df[target].copy()

for c in numeric_cols:
    X[c] = X[c].fillna(X[c].median())

for c in categorical_cols:
    X[c] = X[c].astype("object").fillna("Unknown")
    X[c] = X[c].astype("category").cat.codes

feature_names = numeric_cols + categorical_cols

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RNG)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=RNG)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaler_curve = StandardScaler()
X_tr_scaled = scaler_curve.fit_transform(X_tr)
X_val_scaled = scaler_curve.transform(X_val)

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/3454786828.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[c] = X[c].astype("object").fillna("Unknown")


Train shape: (23457, 19)  Test shape: (5865, 19)


## 5. Baseline: Ordinary Least Squares Linear Regression

In [11]:
t0 = time.time()
lin = LinearRegression()
lin.fit(X_train_scaled, y_train)
lin_time = time.time() - t0
print(f"Linear Regression trained in {lin_time:.4f} s")

Linear Regression trained in 0.0061 s


## 6. Hyperparameter tuning (5-fold cross-validation, grid search)
Ridge, Lasso and Elastic Net are each tuned with `GridSearchCV` over the
search space given in the lab manual, using 5-fold cross-validation scored
on R².

In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)

param_grids = {
    "Ridge Regression": (Ridge(), {"alpha": [0.01, 0.1, 1, 10, 100]}),
    "Lasso Regression": (Lasso(max_iter=200000, tol=1e-2), {"alpha": [0.001, 0.01, 0.1, 1, 10]}),
    "Elastic Net Regression": (
        ElasticNet(max_iter=200000, tol=1e-2),
        {"alpha": [0.01, 0.1, 1, 10], "l1_ratio": [0.2, 0.5, 0.8]},
    ),
}

best_models = {"Linear Regression": lin}
tuning_summary = []
training_times = {"Linear Regression": lin_time}

for name, (estimator, grid) in param_grids.items():
    t0 = time.time()
    gs = GridSearchCV(estimator, grid, scoring="r2", cv=kf, n_jobs=-1)
    gs.fit(X_train_scaled, y_train)
    training_times[name] = time.time() - t0
    best_models[name] = gs.best_estimator_
    tuning_summary.append({"Model": name, "Best Parameters": gs.best_params_, "Best CV R2": gs.best_score_})
    print(name, "-> best params:", gs.best_params_, " best CV R2:", round(gs.best_score_, 4))

tuning_df = pd.DataFrame(tuning_summary)
tuning_df

Ridge Regression -> best params: {'alpha': 100}  best CV R2: 0.594
Lasso Regression -> best params: {'alpha': 10}  best CV R2: 0.5928
Elastic Net Regression -> best params: {'alpha': 0.1, 'l1_ratio': 0.8}  best CV R2: 0.5958


,Model,Best Parameters,Best CV R2
0,Ridge Regression,{'alpha': 100},0.594046
1,Lasso Regression,{'alpha': 10},0.592781
2,Elastic Net Regression,"{'alpha': 0.1, 'l1_ratio': 0.8}",0.595760


## 7. Cross-validation performance
With the best hyperparameters fixed, each model is scored again with 5-fold
cross-validation on the training set to report MAE, MSE, RMSE and R².

In [13]:
scoring = {
    "MAE": make_scorer(mean_absolute_error),
    "MSE": make_scorer(mean_squared_error),
    "R2": make_scorer(r2_score),
}

cv_rows = []
for name, model in best_models.items():
    cvres = cross_validate(model, X_train_scaled, y_train, cv=kf, scoring=scoring, n_jobs=-1)
    mae, mse = cvres["test_MAE"].mean(), cvres["test_MSE"].mean()
    cv_rows.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": np.sqrt(mse), "R2": cvres["test_R2"].mean()})
cv_df = pd.DataFrame(cv_rows)
cv_df

,Model,MAE,MSE,RMSE,R2
0,Linear Regression,21032.038826,9.570083e+08,30935.550396,0.590901
1,Ridge Regression,21052.582168,9.496486e+08,30816.369555,0.594046
2,Lasso Regression,21028.015328,9.526068e+08,30864.329530,0.592781
3,Elastic Net Regression,21117.084443,9.456505e+08,30751.430236,0.595760


## 8. Test set evaluation
Finally, every model is trained on the full training set and evaluated once
on the untouched 20% test split.

In [14]:
test_rows = []
predictions = {}
for name, model in best_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred
    mse = mean_squared_error(y_test, y_pred)
    test_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(y_test, y_pred),
        "Training Time (s)": training_times[name],
    })
test_df = pd.DataFrame(test_rows)
test_df

,Model,MAE,MSE,RMSE,R2,Training Time (s)
0,Linear Regression,20706.011585,8.925525e+08,29875.616759,0.603192,0.006093
1,Ridge Regression,20737.307929,8.934730e+08,29891.018385,0.602783,2.856702
2,Lasso Regression,20707.066263,8.926325e+08,29876.955474,0.603156,0.248120
3,Elastic Net Regression,20839.274657,8.979007e+08,29964.990720,0.600814,0.378397


## 9. Predicted vs Actual, and Residuals

In [15]:
short_name = {"Linear Regression": "linear", "Ridge Regression": "ridge",
              "Lasso Regression": "lasso", "Elastic Net Regression": "elasticnet"}

for name, y_pred in predictions.items():
    tag = short_name[name]
    idx = np.random.RandomState(42).choice(len(y_test), size=min(1200, len(y_test)), replace=False)
    yt, yp = np.array(y_test)[idx], np.array(y_pred)[idx]

    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.scatter(yt, yp, s=10, alpha=0.35, color="#E58A2A")
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, "--", color="#B23A48", linewidth=1.5)
    ax.set_xlabel("Actual Loan Amount (USD)"); ax.set_ylabel("Predicted Loan Amount (USD)")
    ax.set_title(f"{name}: Predicted vs Actual")
    plt.tight_layout()
    plt.savefig(f"images/pred_vs_actual_{tag}.png", bbox_inches="tight")
    plt.show()

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/3858318781.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
for name, y_pred in predictions.items():
    tag = short_name[name]
    residuals = np.array(y_test) - np.array(y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    ridx = np.random.RandomState(1).choice(len(y_pred), size=min(1200, len(y_pred)), replace=False)
    axes[0].scatter(np.array(y_pred)[ridx], residuals[ridx], s=10, alpha=0.35, color="#D46A6A")
    axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[0].set_xlabel("Predicted Loan Amount (USD)"); axes[0].set_ylabel("Residual")
    axes[0].set_title(f"{name}: Residuals")
    sns.histplot(residuals, bins=50, kde=True, color="#D46A6A", ax=axes[1])
    axes[1].set_xlabel("Residual (USD)"); axes[1].set_title("Residual Distribution")
    plt.tight_layout()
    plt.savefig(f"images/residuals_{tag}.png", bbox_inches="tight")
    plt.show()

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/4149078322.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/4149078322.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/4149078322.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/4149078322.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Ridge: training vs validation error across alpha

In [17]:
alphas = np.logspace(-3, 4, 40)
train_rmse, val_rmse = [], []
for a in alphas:
    r = Ridge(alpha=a)
    r.fit(X_tr_scaled, y_tr)
    train_rmse.append(np.sqrt(mean_squared_error(y_tr, r.predict(X_tr_scaled))))
    val_rmse.append(np.sqrt(mean_squared_error(y_val, r.predict(X_val_scaled))))

best_alpha = alphas[int(np.argmin(val_rmse))]
print("Alpha that minimises validation RMSE:", round(best_alpha, 3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(alphas, train_rmse, marker="o", markersize=3, label="Training RMSE", color="#2C6E8E")
ax.semilogx(alphas, val_rmse, marker="o", markersize=3, label="Validation RMSE", color="#E58A2A")
ax.axvline(best_alpha, color="gray", linestyle=":", label=f"Best alpha = {best_alpha:.2g}")
ax.set_xlabel("Alpha (regularization strength)"); ax.set_ylabel("RMSE")
ax.set_title("Ridge Regression: Training vs Validation Error")
ax.legend()
plt.tight_layout()
plt.savefig("images/ridge_train_val_error.png", bbox_inches="tight")
plt.show()

Alpha that minimises validation RMSE: 8.886


/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/4205509213.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Coefficient comparison across models

In [18]:
coef_df = pd.DataFrame({name: model.coef_ for name, model in best_models.items()}, index=feature_names)
top_feats = coef_df.abs().mean(axis=1).sort_values(ascending=False).head(10).index
plot_df = coef_df.loc[top_feats].reset_index().melt(id_vars="index", var_name="Model", value_name="Coefficient")
plot_df = plot_df.rename(columns={"index": "Feature"})

fig, ax = plt.subplots(figsize=(9.5, 4.6))
sns.barplot(data=plot_df, x="Feature", y="Coefficient", hue="Model", ax=ax)
ax.set_title("Standardized Coefficient Comparison Across Models (Top 10 Features)")
ax.tick_params(axis="x", rotation=35)
for label in ax.get_xticklabels():
    label.set_ha("right")
plt.tight_layout()
plt.savefig("images/coefficient_comparison.png", bbox_inches="tight")
plt.show()
coef_df.loc[top_feats]

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/3198789137.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Linear Regression,Ridge Regression,Lasso Regression,Elastic Net Regression
Loan Amount Request (USD),34683.277235,33196.090499,34510.358457,29205.562978
Credit Score,12004.164606,11971.111365,11994.105764,11839.532535
Income Stability,2191.099826,2174.992498,2175.210450,2113.331402
Property Price,-588.920992,704.425248,-434.538294,4067.169239
Current Loan Expenses (USD),-962.403120,-782.009671,-932.462963,-231.695649
Co-Applicant,-681.051894,-684.554429,-671.686202,-690.192517
Type of Employment,665.382499,658.441253,652.618028,634.218567
Expense Type 2,424.241924,420.255552,413.041306,406.884157
No. of Defaults,-386.049516,-385.334897,-376.120662,-381.378586
Dependents,-325.923181,-323.539915,-316.264988,-316.336350


## 12. Shrinkage and Feature Selection
Ridge, Lasso and ElasticNet all shrink coefficients relative to plain OLS, but they do it differently: Ridge shrinks every coefficient a little without ever setting one exactly to zero, while Lasso's L1 penalty can drive individual coefficients all the way to zero, effectively performing feature selection. The plots below compare how much shrinkage each model applies overall, and which specific features Lasso eliminates.

In [19]:
nonzero_counts = (coef_df.abs() > 1e-8).sum(axis=0)
total_features = len(feature_names)
print("Non-zero coefficients out of", total_features, "features:")
print(nonzero_counts)

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ["#8FBFE0", "#2C6E8E", "#E58A2A", "#B23A48"]
bars = ax.bar(nonzero_counts.index, nonzero_counts.values, color=bar_colors[:len(nonzero_counts)])
ax.axhline(total_features, color="gray", linestyle=":", label=f"Total features = {total_features}")
for bar, val in zip(bars, nonzero_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.15, str(val), ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Number of non-zero coefficients")
ax.set_title("Feature Selection: Non-zero Coefficients per Model")
ax.set_xticklabels(nonzero_counts.index, rotation=15, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig("images/feature_selection_counts.png", bbox_inches="tight")
plt.show()


/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/1469913950.py:14: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(nonzero_counts.index, rotation=15, ha="right")


Non-zero coefficients out of 19 features:
Linear Regression         19
Ridge Regression          19
Lasso Regression          17
Elastic Net Regression    19
dtype: int64


/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/1469913950.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
l1_norm = coef_df.abs().sum(axis=0)
l2_norm = np.sqrt((coef_df ** 2).sum(axis=0))
norm_df = pd.DataFrame({"L1 norm (sum |coef|)": l1_norm, "L2 norm (sqrt sum coef^2)": l2_norm})
print(norm_df)

norm_plot = norm_df.reset_index().melt(id_vars="index", var_name="Norm", value_name="Value")
norm_plot = norm_plot.rename(columns={"index": "Model"})

fig, ax = plt.subplots(figsize=(7.5, 4.2))
sns.barplot(data=norm_plot, x="Model", y="Value", hue="Norm", ax=ax, palette=["#6FA8DC", "#B23A48"])
ax.set_title("Coefficient Shrinkage: L1 and L2 Norms Across Models")
ax.set_ylabel("Norm of coefficient vector")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.savefig("images/shrinkage_norms.png", bbox_inches="tight")
plt.show()


                        L1 norm (sum |coef|)  L2 norm (sqrt sum coef^2)
Linear Regression               53884.169680               36805.490262
Ridge Regression                52262.520423               35392.938065
Lasso Regression                53358.925674               36634.423965
Elastic Net Regression          50872.913492               31870.133556


/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/3936731985.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
import matplotlib.patches as mpatches

lasso_coefs = coef_df["Lasso Regression"].sort_values()
point_colors = ["#B23A48" if abs(v) < 1e-8 else "#2C6E8E" for v in lasso_coefs.values]

fig, ax = plt.subplots(figsize=(7, 6))
ax.hlines(y=lasso_coefs.index, xmin=0, xmax=lasso_coefs.values, color=point_colors, linewidth=2)
ax.scatter(lasso_coefs.values, lasso_coefs.index, color=point_colors, s=35, zorder=3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Lasso coefficient (standardized)")
ax.set_title("Lasso Regression: Feature Selection via Coefficient Shrinkage")
handles = [mpatches.Patch(color="#2C6E8E", label="Retained (non-zero)"),
           mpatches.Patch(color="#B23A48", label="Eliminated (= 0)")]
ax.legend(handles=handles, loc="lower right")
plt.tight_layout()
plt.savefig("images/lasso_feature_selection.png", bbox_inches="tight")
plt.show()


/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/2153212697.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Distributions of the four most influential numeric features

In [22]:
key_features = ["Loan Amount Request (USD)", "Property Price",
                 "Current Loan Expenses (USD)", "Credit Score"]
file_tag = {"Loan Amount Request (USD)": "loan_amount_request", "Property Price": "property_price",
            "Current Loan Expenses (USD)": "current_loan_expenses", "Credit Score": "credit_score"}

for col in key_features:
    data = df[col].fillna(df[col].median())
    standardized = (data - data.mean()) / data.std()

    fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
    sns.histplot(data, bins=45, kde=True, color="#8FBFE0", ax=axes[0])
    axes[0].set_title(f"{col}\nOriginal"); axes[0].set_xlabel(col)
    sns.histplot(standardized, bins=45, kde=True, color="#6FA8DC", ax=axes[1])
    axes[1].set_title("Standardized"); axes[1].set_xlabel("z-score")
    sns.boxplot(y=data, color="#B7D6EC", ax=axes[2])
    axes[2].set_title("Spread"); axes[2].set_ylabel(col)
    stats.probplot(standardized, dist="norm", plot=axes[3])
    axes[3].set_title("Q-Q Plot")
    axes[3].get_lines()[0].set_markerfacecolor("#3E7CB1")
    axes[3].get_lines()[0].set_markersize(3)
    plt.tight_layout()
    plt.savefig(f"images/dist_{file_tag[col]}.png", bbox_inches="tight")
    plt.show()

/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/948168889.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/948168889.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/948168889.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn/T/ipykernel_72271/948168889.py:10: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
/var/folders/qf/ng3byxn529d5yz4w_xj9_5lh0000gn

## 14. Saving results

In [23]:
tuning_df.to_csv("tuning_summary.csv", index=False)
cv_df.to_csv("cv_performance.csv", index=False)
test_df.to_csv("test_performance.csv", index=False)
coef_df.to_csv("coefficients.csv")
print("All result tables saved.")

All result tables saved.
